# Shallow Water Layer (single layer)

This is the canonical single-layer model of a constant-density, incompressible fluid.

## Equations

For a single layer, the governing equations are
\begin{align}
\partial_t h + \partial_x \left( h u \right) + \partial_y \left( h v \right) & = 0 \\
\partial_t u - q h v + \partial_x \left( M + K \right) & = \frac{\tau^x - \epsilon u}{h} + \frac{1}{h} \left[ \partial_x \left( \nu_h h \mathcal{T} \right) + \partial_y \left( \nu_h h \mathcal{S} \right) \right] \\
\partial_t v + q h u + \partial_y \left( M + K \right) & = \frac{\tau^y - \epsilon v}{h} + \frac{1}{h} \left[ \partial_x \left( \nu_h h \mathcal{S} \right) - \partial_y \left( \nu_h h \mathcal{T} \right) \right]
\end{align}
where
\begin{align}
\eta & = -D + h \\
M & = g \eta \\
K & = \frac{1}{2} \left( u^2 + v^2 \right) \\
q & = \frac{ f + \zeta }{ h } \\
\zeta & = \partial_x v - \partial_y u \\
f & = f_o + \beta y \\
\mathcal{T} & = \partial_x u - \partial_y v \\
\mathcal{S} & = \partial_y u + \partial_x v
\end{align}
Here $\mathcal{T}$ is the tension (stretching deformation) and $\mathcal{S}$ is the shearing deformation. The viscous terms are the divergence of the horizontal viscous stress tensor divided by $h$. For constant $h$ and $\nu$, these reduce to $\nu \nabla^2 u$ and $\nu \nabla^2 v$.

## Algorithm

$u$, $v$ and $h$ will be staggered on an Arakawa C-grid using south-west indexing convention, i.e. $u_{i-1/2,j}=u[j,i]$ and $v_{i,j-1/2}=v[j,i]$ are west and south of $h_{i,j}=h[j,i]$ respectively.

Time discretization and algorithm (only partial spatial discretization so far, $\tilde{ \cdot }$ indicates upwinding of some form):
\begin{align}
H^u & = \tilde{ h }^i u^n \\
h^* &= h^n - \frac{\Delta t}{\Delta x} \delta_i H^u \\
H^v & = \tilde{ h^* }^j v^n \\
h^{n+1} &= h^* - \frac{\Delta t}{\Delta y} \delta_j H^v \\
K_{i,j}^n & = \max(0,u_{i-1/2,j}^n)^2 + \min(0,u_{i+1/2,j}^n)^2 + \max(0,v_{i,j-1/2}^n)^2 + \min(0,v_{i,j+1/2}^n)^2 \\
q & = \frac{ f + \delta_i ( v^n \Delta y ) - \delta_j ( u^n \Delta x ) }{ h^{n} \Delta x \Delta y } \\
\eta^{n+1} &= -D + h^{n+1} \\
M^{n+1} &= g \eta^{n+1}
\end{align}
and a simultaneous solve of

\begin{align}
\frac{ u^{n+1} - u^n }{ \Delta t}
& = f \left( \alpha_f v^{n+1} + ( 1 - \alpha_f ) v^n \right) +
\left( \tilde{q}^{j} \overline{ H^v }^{ij} - f v^n \right) - \frac{1}{\Delta x} \delta_i \left( M^{n+1} + K^n \right)
+ \frac{\tau^x}{h} - \frac{\epsilon \left( \alpha_\epsilon u^{n+1} + ( 1 - \alpha_\epsilon ) u^n \right)}{h} + \mathcal{F}^x_\nu \\
\frac{ v^{n+1} - v^n }{ \Delta t}
& = - f \left( \alpha_f u^{n+1} + ( 1 - \alpha_f ) u^n \right) +
\left( - \tilde{q}^{i} \overline{ H^u }^{ij} + f u^n \right) - \frac{1}{\Delta y} \delta_j \left( M^{n+1} + K^n \right)
+ \frac{\tau^y}{h} - \frac{\epsilon \left( \alpha_\epsilon v^{n+1} + ( 1 - \alpha_\epsilon ) v^n \right)}{h} + \mathcal{F}^y_\nu
\end{align}
where $\mathcal{F}^x_\nu$ and $\mathcal{F}^y_\nu$ denote the viscous stress tensor divergence (see equations above), evaluated at time level $n$.

Note that the linear Coriolis term, $f v$ is included in the non-linear term, $q H^v$, so we have a compensation term added that let's us treat some part of the Coriolis term implicitly in time.
The parameter $\alpha_f$ allows the Coriolis term to be centered in time ($\alpha=1/2$), preserving oscillation amplitudes, or made Euler-backward ($\alpha=1$) in time which damps oscillations without changing the balanced state.
Similarly, $\alpha_\epsilon$ allows the dissipation to be either explicit ($\alpha_\epsilon=0$) or Euler-backward ($\alpha_\epsilon=1$).

### Implicit-Coriolis and drag

Solution of the latter is managed by re-writing in terms of the velocity increment
\begin{align}
\Delta u &= u^{n+1} - u^n \\
\Delta v &= v^{n+1} - v^n
\end{align}
so the coupled problem becomes
\begin{align}
\left( \frac{1}{\Delta t} + \frac{ \alpha_\epsilon \epsilon }{h} \right) \Delta u - f \alpha_f \Delta v
&= \dot{u} \\
\left( \frac{1}{\Delta t} + \frac{ \alpha_\epsilon \epsilon }{h} \right) \Delta v + f \alpha_f \Delta u
&= \dot{v}
\end{align}
where the $\dot{u}$ and $\dot{v}$ are all accelerations evaluated explicitly:
$$
\begin{aligned}
\dot{u}
& = \tilde{q}^{j} \overline{ H^v }^{ij} - \frac{1}{\Delta x} \delta_i \left( M^{n+1} + K^n \right)
+ \frac{\tau^x}{h} - \frac{\epsilon u^{n}}{h} + \mathcal{F}^x_\nu \\
\dot{v}
& = - \tilde{q}^{i} \overline{ H^u }^{ij} - \frac{1}{\Delta y} \delta_j \left( M^{n+1} + K^n \right)
+ \frac{\tau^y}{h} - \frac{\epsilon v^{n}}{h} + \mathcal{F}^y_\nu \\
\end{aligned}
$$
Rearranging for $\Delta u$ and $\Delta v$, we get
\begin{align}
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{h} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta u
&= \Delta t^2 f \alpha_f \dot{v} + \Delta t \left( 1 + \frac{ \Delta t \alpha_\epsilon\epsilon }{h} \right)\dot{u} \\
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{h} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta v
&= - \Delta t^2  f \alpha_f \dot{u} + \Delta t \left( 1 + \frac{ \Delta t \alpha_\epsilon\epsilon }{h} \right)\dot{v} \\
\end{align}

Finally we apply the spatial discretization (interpolation) needed for the Coriolis terms:
\begin{align}
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{h} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta u
&= \Delta t \left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{h} \right)\dot{u} + \Delta t f \alpha_f \overline{ \dot{v} }^{ij} \right) \\
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{h} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta v
&= \Delta t \left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{h} \right)\dot{v} - \Delta t f \alpha_f \overline{ \dot{u} }^{ij} \right)
\end{align}

# Stacked Shallow Water Layers (multiple layers)

These are stacked layers of immiscible fluid, each essentially governed by shallow water layer dynamics except for coupling through pressure between layers.

## Equations

Governing equations are
\begin{align}
\partial_t h_k + \partial_x \left( h_k u_k \right) + \partial_y \left( h_k v_k \right) & = 0 \\
\partial_t u_k - q_k h_k v_k + \partial_x \left( M_k + K_k \right) & = \frac{\tau^u_{k-1/2} - \tau^u_{k+1/2}}{h_k} + \frac{1}{h_k} \left[ \partial_x \left( \nu_h h_k \mathcal{T}_k \right) + \partial_y \left( \nu_h h_k \mathcal{S}_k \right) \right] \\
\partial_t v_k + q_k h_k u_k + \partial_y \left( M_k + K_k \right) & = \frac{\tau^v_{k-1/2} - \tau^v_{k+1/2}}{h_k} + \frac{1}{h_k} \left[ \partial_x \left( \nu_h h_k \mathcal{S}_k \right) - \partial_y \left( \nu_h h_k \mathcal{T}_k \right) \right]
\end{align}
for $k \in \left\{ 1 \ldots K \right\} $. Here
\begin{align}
\eta_{k-1/2} & = -D + \sum_{l=k}^{K} h_l \\
M_k & = \sum_{l=1}^k g'_l \eta_{l-1/2} \\
K_k & = \frac{1}{2} \left( u_k^2 + v_k^2 \right) \\
q_k & = \frac{ f + \zeta_k }{ h_k } \\
\zeta_k & = \partial_x v_k - \partial_y u_k \\
f & = f_o + \beta y \\
\mathcal{T}_k & = \partial_x u_k - \partial_y v_k \\
\mathcal{S}_k & = \partial_y u_k + \partial_x v_k \\
\tau^u_{k-1/2} & = \nu_v \frac{u_{k-1} - u_k}{\frac{1}{2} ( h_{k-1} + h_k )} \\
\tau^v_{k-1/2} & = \nu_v \frac{v_{k-1} - v_k}{\frac{1}{2} ( h_{k-1} + h_k )}
\end{align}
and top boundary conditions
\begin{align}
\tau^u_{1/2} &= \tau^x \\
\tau^v_{1/2} &= \tau^y
\end{align}
and bottom boundary conditions
\begin{align}
\tau^u_{K+1/2} &= \epsilon u_K \\
\tau^v_{K+1/2} &= \epsilon v_K
\end{align}
As in the single-layer case, $\mathcal{T}_k$ is the tension (stretching deformation) and $\mathcal{S}_k$ is the shearing deformation of layer $k$. The horizontal viscous terms are the divergence of the per-layer horizontal viscous stress tensor divided by $h_k$; for constant $h_k$ and $\nu_h$ they reduce to $\nu_h \nabla^2 u_k$ and $\nu_h \nabla^2 v_k$.

$u$, $v$ and $h$ will be staggered on an Arakawa C-grid using south-west indexing convention in the horizontal, and with layer index $k$ increasing downwards. However, the math has the top layer at $k=1$ while python is 0-indexed so the top layer is $[0]$. What matters is the relative stagger, i.e. $u_{i-1/2,j,k}=u[k,j,i]$ and $v_{i,j-1/2,k}=v[k,j,i]$ are west and south of $h_{i,j,k}=h[k,j,i]$ respectively. The surface is $\eta_{i,j,1/2}=eta[0,j,i]$ and since $\eta_{i,j,k-1/2}$ is above $h_{i,j,k}$ then $eta[k,j,i]$ is above $h[k,j,i]$.

## Algorithm

The continuity, kinetic-energy, vorticity, pressure-gradient, and horizontal-viscous-divergence steps proceed per layer exactly as in the single-layer algorithm above (with $h \to h_k$, $u \to u_k$, etc.; $M_k$ now comes from the per-layer Montgomery potential defined in the equations above). What changes is the treatment of the velocity-dependent interfacial stresses, which now appear in every layer and couple adjacent layers. Bottom drag is no longer a stand-alone term: it is the bottom-boundary value of the same vertical-diffusion operator, and we treat it with the same time weighting.

Collect the interfacial-stress coefficients at each interface $k\pm 1/2$:
\begin{align}
a_{1/2} &= 0 & & \text{(top stress is wind, treated explicitly)} \\
a_{k-1/2} &= \frac{2 \nu_v}{h_{k-1}+h_k} & & \text{for } 1 < k \le K \text{ (interior interfaces)} \\
a_{K+1/2} &= \epsilon & & \text{(bottom drag is the bottom-boundary stress)}
\end{align}
so that $\tau^u_{k-1/2} = a_{k-1/2}(u_{k-1}-u_k)$ for interior interfaces, $\tau^u_{1/2} = \tau^x$, and $\tau^u_{K+1/2} = a_{K+1/2}\, u_K$. The combined vertical-viscosity + bottom-drag contribution to $\partial_t u_k$ is then
\begin{align}
\frac{\tau^u_{k-1/2} - \tau^u_{k+1/2}}{h_k} & = \frac{\tau^x \delta_{k,1}}{h_k} - (L\,u)_k
\end{align}
where $L$ is the per-column tridiagonal vertical-diffusion operator
\begin{align}
(L\,u)_k & = \frac{a_{k-1/2}+a_{k+1/2}}{h_k}\, u_k - \frac{a_{k-1/2}}{h_k}\, u_{k-1} - \frac{a_{k+1/2}}{h_k}\, u_{k+1}
\end{align}
with the convention that the $u_{k-1}$ term vanishes for $k=1$ (because $a_{1/2}=0$) and the $u_{k+1}$ term vanishes for $k=K$ (because there is no layer $K+1$; the $a_{K+1/2}\,u_K$ contribution remains on the diagonal as bottom drag). For $K=1$, $L$ collapses to the scalar $\epsilon/h$, recovering the single-layer drag.

We treat $L\,u$ with the same Euler-backward weighting that the single-layer drag had, introducing the parameter $\alpha_\nu$ (which subsumes $\alpha_\epsilon$ since bottom drag is just the bottom entry of $L$): $L u \to \alpha_\nu\, L u^{n+1} + (1-\alpha_\nu)\, L u^n$. The wind $\tau^x/h_1$ stays explicit. Working in terms of $\Delta u_k = u_k^{n+1} - u_k^n$ and $\Delta v_k = v_k^{n+1} - v_k^n$, the coupled implicit problem becomes
\begin{align}
\left( \frac{1}{\Delta t} I + \alpha_\nu L \right) \Delta u - f \alpha_f \Delta v &= \dot{u} \\
\left( \frac{1}{\Delta t} I + \alpha_\nu L \right) \Delta v + f \alpha_f \Delta u &= \dot{v}
\end{align}
where the explicit accelerations now contain the per-layer terms plus the full time-$n$ value of the interfacial stresses:
$$
\begin{aligned}
\dot{u}_k & = \tilde{q}_k^{j} \overline{ H^v_k }^{ij}
- \frac{1}{\Delta x} \delta_i \left( M_k^{n+1} + K_k^n \right)
+ \frac{\tau^x \delta_{k,1}}{h_k}
- (L\,u^n)_k + \mathcal{F}^x_{\nu_h,k} \\
\dot{v}_k & = - \tilde{q}_k^{i} \overline{ H^u_k }^{ij}
- \frac{1}{\Delta y} \delta_j \left( M_k^{n+1} + K_k^n \right)
+ \frac{\tau^y \delta_{k,1}}{h_k}
- (L\,v^n)_k + \mathcal{F}^y_{\nu_h,k}
\end{aligned}
$$
For $K=1$ the $-(L\,u^n)_k$ term reduces to $-\epsilon u^n/h$ and the system reduces to the single-layer one above. The wind $\tau^x \delta_{k,1}/h_k$ now lives only on the top layer (and is the explicit counterpart of what was previously folded into the "single layer = top layer" case).

The parameter $\alpha_\nu$ plays the role $\alpha_\epsilon$ played in the single-layer algorithm: $\alpha_\nu = 0$ gives fully explicit interfacial stresses (and inherits the stiff viscous CFL constraint $\nu_v \Delta t / h_k^2 \lesssim O(1)$ plus the drag constraint $\epsilon \Delta t / h_K \lesssim O(1)$), while $\alpha_\nu = 1$ is Euler-backward and removes both constraints at the cost of one tridiagonal solve per column.

### Implicit solver — row-scaled cancellation-free Thomas (TDMAH2)

The coupled system written above is straightforward to solve with the complex substitution $\Delta w = \Delta u + i\, \Delta v$, which collapses the $u/v$ Coriolis cross-coupling and reduces the per-column problem to a single $K$-element complex tridiagonal:
\begin{align}
(A + i\, c\, I)\, \Delta w = r_w
\end{align}
with $A \equiv I + \alpha_\nu \Delta t\, L$, $c \equiv \alpha_f \Delta t\, f$, and $r_w \equiv \Delta t\, ( \dot{u} + i\, \overline{ \dot{v} }^{ij} )$ at u-points (and analogously at v-points). See `Implicit-step-derivation.ipynb` for the step-by-step algebra.

As written, $A$ carries factors of $1/h_k$ everywhere because the momentum equation has the $(L u)_k$ term divided by $h_k$ — so the matrix is also asymmetric ($A_{k,k+1} \ne A_{k+1,k}$ unless $h_k = h_{k+1}$) and ill-conditioned as $h_k \to 0$. Both problems are removed by **row-scaling each equation by $h_k$**. Letting $a^*_{k\pm 1/2} \equiv \alpha_\nu \Delta t\, a_{k\pm 1/2}$ (real, $\ge 0$; with $a^*_{1/2} = 0$ and $a^*_{K+1/2} = \alpha_\nu \Delta t\, \epsilon$), the row-scaled tridiagonal has entries
\begin{align}
b_k &= h_k(1 + i\, c) + a^*_{k-1/2} + a^*_{k+1/2} & & \text{(diagonal)} \\
a_k &= -a^*_{k-1/2} & & \text{(sub-diagonal, } k \ge 1 \text{)} \\
c_k &= -a^*_{k+1/2} & & \text{(super-diagonal, } k \le K-2 \text{)}
\end{align}
which is now **symmetric** ($a_{k+1} = c_k$), free of $1/h_k$ factors, and stays well-conditioned as $h_k \to 0$ (in that limit the diagonal goes to $a^*_{k-1/2} + a^*_{k+1/2}$, finite; the RHS scales as $h_k r_w$ and goes to zero, so $\Delta w_k$ degenerates to the neighbour's value, which is the physically correct behaviour for a vanishing layer). The row-scaled right-hand side is
\begin{align}
y_k = h_k\, \Delta t\, (\dot{u}_k + i\, \overline{ \dot{v} }^{ij}_k)
\end{align}
at u-points (analogous at v-points).

We solve this with Hallberg's cancellation-free TDMAH2 variant of Thomas. Define $q_k \equiv a^*_{k+1/2} \beta_k$ and the running ratio $Q_k \equiv (h_k(1+ic) + a^*_{k-1/2}\, Q_{k-1})\, \beta_k$ where $\beta_k \equiv 1 / (h_k(1+ic) + a^*_{k-1/2}\, Q_{k-1} + a^*_{k+1/2})$. The forward sweep is
\begin{align}
\beta_0 &= 1\, /\, ( h_0 (1 + ic) + a^*_{1/2} + a^*_{3/2} ) \\
q_0 &= a^*_{3/2}\, \beta_0 \\
Q_0 &= h_0 (1 + ic)\, \beta_0 \\
y'_0 &= h_0\, \Delta t\, (\dot{u}_0 + i\, \overline{\dot{v}}^{ij}_0)\, \beta_0
\end{align}
and for $k = 1, \dots, K-1$:
\begin{align}
\beta_k &= 1\, /\, ( h_k (1+ic) + a^*_{k-1/2}\, Q_{k-1} + a^*_{k+1/2} ) \\
q_k &= a^*_{k+1/2}\, \beta_k \\
Q_k &= ( h_k (1+ic) + a^*_{k-1/2}\, Q_{k-1} )\, \beta_k \\
y'_k &= ( h_k\, \Delta t\, (\dot{u}_k + i\, \overline{\dot{v}}^{ij}_k) + a^*_{k-1/2}\, y'_{k-1} )\, \beta_k
\end{align}
followed by back substitution
\begin{align}
\Delta w_{K-1} &= y'_{K-1} \\
\Delta w_k &= y'_k + q_k\, \Delta w_{k+1} \qquad (k = K-2, \dots, 0)
\end{align}
and finally $\Delta u_k = \mathrm{Re}(\Delta w_k)$ at u-points (or $\Delta v_k = \mathrm{Im}(\Delta w_k)$ at v-points, from the analogous v-point solve).

The key property is that every denominator $\beta_k^{-1}$ is a sum of three nonnegative-real or complex-with-positive-real-part terms — no subtractions of nearly equal positive quantities, so no catastrophic cancellation. Back-substitution likewise uses $+$ rather than $-$ (because we work with $q_k \equiv -c'_k$ where $c'_k$ is the standard Thomas modified super-diagonal). The algorithm is therefore robust as layers vanish; in the OSSWEM code (`_step_numba` in `OSSWEM.py`) the per-$k$ recurrence is implemented as $(n_j, n_i)$ complex array operations, so the column problems are solved simultaneously over all $(i, j)$.

For $K = 1$, $a^*_{1/2} = 0$, $a^*_{3/2} = \alpha_\nu \Delta t\, \epsilon$, the forward sweep stops at $k = 0$, and $\Delta w_0 = y'_0 = h_0\, \Delta t\, (\dot u + i \overline{\dot v})\, /\, ( h_0 (1 + ic) + \alpha_\nu \Delta t\, \epsilon )$. Dividing top and bottom by $h_0$:
$$ \Delta w_0 = \frac{\Delta t\, (\dot u + i\, \overline{\dot v})}{(1 + \alpha_\nu \Delta t\, \epsilon / h_0) + ic} $$
whose real part is exactly the single-layer closed-form $\Delta u$ from the previous section.